# 12 — Transformer Blocks and GPT

## Goal

Previous lessons studied the main components of a decoder-only
Transformer independently:

- token embeddings,
- causal self-attention,
- multi-head attention,
- RMSNorm,
- residual connections,
- feed-forward networks,
- SwiGLU,
- and Rotary Position Embeddings.

This lesson assembles those components into a complete decoder-only
language model.

The main objective is to understand how information flows through the
entire model:

$$
\text{token IDs}
\rightarrow
\text{embeddings}
\rightarrow
\text{Transformer blocks}
\rightarrow
\text{final normalization}
\rightarrow
\text{language-model head}
\rightarrow
\text{logits}.
$$

The focus is no longer on reimplementing mechanisms that have already
been studied.

Instead, the focus is on **composition**:

- how modules connect,
- which tensor shapes must be preserved,
- where RoPE is applied,
- how residual streams flow through depth,
- and how the final hidden states become next-token logits.

In [2]:
from dataclasses import dataclass
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

## 1. The Decoder-Only Transformer

A decoder-only Transformer maintains a residual stream with shape

$$
X \in \mathbb{R}^{B \times T \times C}.
$$

The model begins with token IDs:

$$
(B,T),
$$

which are converted into token embeddings:

$$
(B,T)
\rightarrow
(B,T,C).
$$

The representation then passes through a stack of Transformer blocks.

Each block preserves the same residual-stream shape:

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

After the final block, a normalization layer is applied, followed by a
language-model head:

$$
(B,T,C)
\rightarrow
(B,T,V),
$$

where $V$ is the vocabulary size.

The output tensor contains one vocabulary-logit vector for every token
position.

![Transformer Block](./imgs/transformer_block.png)


In [3]:
from typing import Any


class TransformerBlock(nn.Module):
    """A pre-norm decoder-only Transformer block.

    The block contains two residual sublayers:

    1. RMSNorm followed by causal self-attention.
    2. RMSNorm followed by a feed-forward network.

    Both sublayers preserve the residual-stream shape `(B, T, C)`.

    Args:
        embedding_dim: Width of the residual stream.
        attention: Causal self-attention module.
        feed_forward: Token-wise feed-forward module."""

    def __init__(
        self, embedding_dim: int, attention: nn.Module, feed_forward: nn.Module
    ) -> None:
        super().__init__()
        self.attention_norm = nn.RMSNorm(embedding_dim)
        self.attention = attention
        self.feed_forward_norm = nn.RMSNorm(embedding_dim)

        self.feed_forward = feed_forward

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply attention and feed-forward residual updates.

        Args:
            x: Residual-stream tensor of shape `(B, T, C)`.

        Returns:
            Updated residual-stream tensor with shape `(B, T, C)`.
        """
        # The original residual stream bypasses the attention sublayer.
        x = self.attention(self.attention_norm(x))
        # The updated residual stream then bypasses the MLP sublayer.
        x = x + self.feed_forward(self.feed_forward_norm(x))

        return x


## 2. RoPE-Aware Causal Self-Attention

Previous lessons implemented causal attention and RoPE separately.

We now combine them into a reusable attention module.

The input is the residual-stream representation

$$
X \in \mathbb{R}^{B \times T \times C}.
$$

Query, key, and value projections preserve the total model width:

$$
Q,K,V
\in
\mathbb{R}^{B \times T \times C}.
$$

They are then split into $H$ attention heads:

$$
(B,T,C)
\rightarrow
(B,H,T,D),
$$

where

$$
D=\frac{C}{H}.
$$

RoPE is applied to queries and keys before the attention scores are
computed.

The attention output is finally concatenated back into

$$
(B,T,C)
$$

and projected into the residual stream.

In [4]:
def rope_frequencies(
    head_dim: int,
    base: float = 10000.0,
    *,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Create the angular frequencies used by Rotary Position Embeddings.

    Args:
        head_dim: Feature dimension of one attention head.
        base: Base controlling the geometric spacing of frequencies.
        device: Device on which to create the frequency tensor.

    Returns:
        A tensor of shape `(head_dim / 2,)`, containing one angular
        frequency for each adjacent feature pair.

    Raises:
        ValueError: If `head_dim` is not even.
    """
    if head_dim % 2 != 0:
        raise ValueError("head_dim must be even for RoPE.")

    # Every adjacent feature pair shares one rotation frequency.
    dimension_indices: torch.Tensor = torch.arange(
        0,
        head_dim,
        2,
        dtype=torch.float32,
        device=device,
    )

    return torch.pow(
        base,
        -dimension_indices / head_dim,
    )

In [5]:
def build_rope_cache(
    sequence_length: int,
    head_dim: int,
    *,
    base: float = 10000.0,
    device: torch.device | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Precompute cosine and sine values used by RoPE.

    Args:
        sequence_length: Number of token positions.
        head_dim: Feature dimension of one attention head.
        base: Base controlling the range of RoPE frequencies.
        device: Device on which to create the tensors.

    Returns:
        A pair `(cos_values, sin_values)`, each with shape
        `(sequence_length, head_dim / 2)`.
    """
    frequencies: torch.Tensor = rope_frequencies(
        head_dim=head_dim,
        base=base,
        device=device,
    )

    positions: torch.Tensor = torch.arange(
        sequence_length,
        dtype=torch.float32,
        device=device,
    )

    # Create one rotation angle for every (position, frequency) pair.
    angles: torch.Tensor = positions.unsqueeze(1) * frequencies.unsqueeze(0)

    return (
        torch.cos(angles),
        torch.sin(angles),
    )

In [6]:
def apply_rope(
    x: torch.Tensor,
    cos_values: torch.Tensor,
    sin_values: torch.Tensor,
) -> torch.Tensor:
    """Apply RoPE to an attention tensor.

    Args:
        x: Tensor of shape `(B, H, T, D)`.
        cos_values: Cosine values of shape `(T, D / 2)`.
        sin_values: Sine values of shape `(T, D / 2)`.

    Returns:
        Rotated tensor with the same shape `(B, H, T, D)`.
    """
    x_even: torch.Tensor = x[..., 0::2]
    x_odd: torch.Tensor = x[..., 1::2]

    # Add batch and head axes so the same positional rotations
    # broadcast across all examples and attention heads.
    cos_values = cos_values.unsqueeze(0).unsqueeze(0)
    sin_values = sin_values.unsqueeze(0).unsqueeze(0)

    rotated_even: torch.Tensor = x_even * cos_values - x_odd * sin_values

    rotated_odd: torch.Tensor = x_even * sin_values + x_odd * cos_values

    # Reconstruct adjacent rotated feature pairs.
    rotated_pairs: torch.Tensor = torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    )

    return rotated_pairs.flatten(start_dim=-2)

In [7]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with Rotary Position Embeddings.

    Args:
        embedding_dim: Width of the residual stream.
        num_heads: Number of attention heads.
        rope_base: Base used to construct RoPE frequencies.
    """

    def __init__(
        self, embedding_dim: int, num_heads: int, rope_base: float = 10000.0
    ) -> None:
        super().__init__()
        if embedding_dim % num_heads != 0:
            raise ValueError("embedding_dim mut be divisible by num_heads")

        self.embedding_dim: int = embedding_dim
        self.num_heads: int = num_heads
        self.head_num: int = embedding_dim // num_heads
        self.rope_base: float = rope_base

        if self.head_num % 2 != 0:
            raise ValueError("head_dim must be even for RoPE")

        self.query_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )
        self.key_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )
        self.value_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )
        self.output_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply RoPE-aware causal self-attention.

        Args:
            x: Residual-stream tensor of shape `(B, T, C)`.

        Returns:
            Attention output with shape `(B, T, C)`.
        """
        _, sequence_length, _ = x.shape

        q: torch.Tensor = self.query_projection(x)
        k: torch.Tensor = self.key_projection(x)
        v: torch.Tensor = self.value_projection(x)

        # split heads
        q = rearrange(q, "b t (h d) -> b h t d", h=self.num_heads)
        k = rearrange(k, "b t (h d) -> b h t d", h=self.num_heads)
        v = rearrange(v, "b t (h d) -> b h t d", h=self.num_heads)

        # build RoPE cache
        cos_values, sin_values = build_rope_cache(
            sequence_length=sequence_length,
            head_dim=self.head_num,
            base=self.rope_base,
            device=x.device,
        )

        # apply RoPE only to Q and K
        q = apply_rope(q, cos_values, sin_values)
        k = apply_rope(k, cos_values, sin_values)

        # attention
        # scores = q @ k.transpose(-2, -1)
        # scores /= torch.sqrt(d)
        # mask -> softmax -> weights @ V -> output

        attended: torch.Tensor = F.scaled_dot_product_attention(
            q, k, v, is_causal=True
        )
        # Concatenate all attention heads back into model width C.
        attended = rearrange(
            attended,
            "b h t d -> b t (h d)",
        )

        return self.output_projection(attended)

### Using an Optimized Attention Primitive

Earlier lessons implemented scaled dot-product attention explicitly.

At this stage, the mechanism is already understood, so the model uses

```py
F.scaled_dot_product_attention(...)
```

as the attention primitive.

This preserves the learning principle:

implement a mechanism explicitly when it is new; use a mature
implementation after its tensor semantics and mathematics are
understood.

The surrounding architecture — projections, head organization, RoPE,
residual structure, and block composition — remains explicit.


---

# 10. Concatenate heads

```py
        attended = rearrange(
            attended,
            "b h t d -> b t (h d)",
        )
```

In [8]:
# shape test
attention = CausalSelfAttention(
    embedding_dim=32,
    num_heads=4,
)

x: torch.Tensor = torch.randn(
    2,
    10,
    32,
)

output: torch.Tensor = attention(x)

print("input:", x.shape)
print("output:", output.shape)

input: torch.Size([2, 10, 32])
output: torch.Size([2, 10, 32])


In [9]:
# causality verification
attention.eval()

x_original: torch.Tensor = torch.randn(
    1,
    6,
    32,
)

x_modified: torch.Tensor = x_original.clone()

# Change only future tokens.
x_modified[:, 4:, :] = torch.randn_like(x_modified[:, 4:, :])

with torch.no_grad():
    output_original: torch.Tensor = attention(x_original)

    output_modified: torch.Tensor = attention(x_modified)

print(
    "Earlier positions unchanged:",
    torch.allclose(
        output_original[:, :4, :],
        output_modified[:, :4, :],
        atol=1e-5,
    ),
)

Earlier positions unchanged: True


## 3. SwiGLU Feed-Forward Network

The feed-forward sublayer transforms each token independently along the
feature dimension.

This model uses a SwiGLU-style gated feed-forward network:

$$
g = \operatorname{SiLU}(XW_{\text{gate}})
$$

$$
u = XW_{\text{up}}
$$

$$
h = g \odot u
$$

$$
Y = hW_{\text{down}}.
$$

The shape transformation is

$$
(B,T,C)
\rightarrow
(B,T,M)
\rightarrow
(B,T,C),
$$

where $M$ is the hidden dimension of the feed-forward network.

Unlike attention, the feed-forward network does not mix information
between token positions.

In [10]:
class SwiGLU(nn.Module):
    """SwiGLU feed-forward network for a Transformer block.

    The module uses two parallel input projections. One branch is passed
    through SiLU and acts as a gate for the other branch.

    Args:
        embedding_dim: Width of the residual stream.
        hidden_dim: Width of the intermediate feed-forward representation.
    """

    def __init__(
        self,
        embedding_dim: int,
        hidden_dim: int,
    ) -> None:
        super().__init__()

        self.gate_projection = nn.Linear(
            embedding_dim,
            hidden_dim,
            bias=False,
        )

        self.up_projection = nn.Linear(
            embedding_dim,
            hidden_dim,
            bias=False,
        )

        self.down_projection = nn.Linear(
            hidden_dim,
            embedding_dim,
            bias=False,
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        """Apply the SwiGLU transformation.

        Args:
            x: Input tensor of shape `(B, T, C)`.

        Returns:
            Output tensor with the same shape `(B, T, C)`.
        """
        # The gate branch learns how strongly each hidden feature
        # should contribute.
        gate: torch.Tensor = F.silu(self.gate_projection(x))

        # The up branch contains the candidate hidden features.
        content: torch.Tensor = self.up_projection(x)

        # Element-wise multiplication performs learned feature gating.
        hidden: torch.Tensor = gate * content

        # Project back to the residual-stream width.
        return self.down_projection(hidden)

In [11]:
# shape test
feed_forward = SwiGLU(
    embedding_dim=32,
    hidden_dim=96,
)

x: torch.Tensor = torch.randn(
    2,
    10,
    32,
)

output: torch.Tensor = feed_forward(x)

print("input:", x.shape)
print("output:", output.shape)

input: torch.Size([2, 10, 32])
output: torch.Size([2, 10, 32])


In [12]:
attention = CausalSelfAttention(
    embedding_dim=32,
    num_heads=4,
)

feed_forward = SwiGLU(
    embedding_dim=32,
    hidden_dim=96,
)

block = TransformerBlock(
    embedding_dim=32,
    attention=attention,
    feed_forward=feed_forward,
)

x: torch.Tensor = torch.randn(
    2,
    10,
    32,
)

output: torch.Tensor = block(x)

print("input:", x.shape)
print("output:", output.shape)

input: torch.Size([2, 10, 32])
output: torch.Size([2, 10, 32])


## 4. Stacking Transformer Blocks

A Transformer language model does not use a single Transformer block.

Instead, it applies a sequence of blocks:

$$
x_0
\rightarrow
x_1
\rightarrow
x_2
\rightarrow
\cdots
\rightarrow
x_N.
$$

Each block preserves the residual-stream shape:

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

Therefore, the output of one block can be passed directly into the next.

Each block usually has its own parameters, even though all blocks share
the same architecture.

### Why `nn.ModuleList`?

A normal Python list can store modules, but PyTorch needs to know which
objects are registered submodules of the model.

`nn.ModuleList` behaves like a Python list while also registering every
contained module.

This means that operations such as

```py
model.parameters()
model.state_dict()
model.to(device)
model.train()
model.eval()
```

automatically include every Transformer block.


---

In [13]:
num_layers: int = 3
embedding_dim: int = 32
num_heads: int = 4
hidden_dim: int = 96

blocks = nn.ModuleList(
    [
        TransformerBlock(
            embedding_dim=embedding_dim,
            attention=CausalSelfAttention(
                embedding_dim=embedding_dim,
                num_heads=num_heads,
            ),
            feed_forward=SwiGLU(
                embedding_dim=embedding_dim,
                hidden_dim=hidden_dim,
            ),
        )
        for _ in range(num_layers)
    ]
)

In [14]:
print(blocks[0] is blocks[1])
print(
    blocks[0].attention.query_projection.weight
    is blocks[1].attention.query_projection.weight
)

False
False


In [15]:
# shape test

x: torch.Tensor = torch.randn(
    2,
    10,
    embedding_dim,
)

print("input:", x.shape)

for layer_index, block in enumerate(blocks):
    x = block(x)

    print(
        f"after block {layer_index}:",
        x.shape,
    )

input: torch.Size([2, 10, 32])
after block 0: torch.Size([2, 10, 32])
after block 1: torch.Size([2, 10, 32])
after block 2: torch.Size([2, 10, 32])


In [16]:
x: torch.Tensor = torch.randn(
    1,
    4,
    embedding_dim,
)

representations: list[torch.Tensor] = [x]

for block in blocks:
    x = block(x)
    representations.append(x)

for layer_index in range(len(representations) - 1):
    unchanged: bool = torch.allclose(
        representations[layer_index],
        representations[layer_index + 1],
    )

    print(
        f"layer {layer_index} unchanged:",
        unchanged,
    )

layer 0 unchanged: False
layer 1 unchanged: False
layer 2 unchanged: False


In [17]:
def count_parameters(
    module: nn.Module,
) -> int:
    """Count all trainable parameters in a PyTorch module.

    Args:
        module: Module whose parameters should be counted.

    Returns:
        Total number of trainable scalar parameters.
    """
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


print(
    "one block:",
    count_parameters(blocks[0]),
)

print(
    "three blocks:",
    count_parameters(blocks),
)

one block: 13376
three blocks: 40128


## 5. GPT Configuration

As the model grows, several architectural hyperparameters need to remain
consistent across different modules.

Examples include:

- vocabulary size,
- residual-stream width,
- number of attention heads,
- number of Transformer layers,
- feed-forward hidden dimension,
- and the RoPE frequency base.

Instead of passing these values independently throughout the model, we
store them in a configuration object.

The configuration contains model structure, while the model itself
contains learned parameters.

In [18]:
@dataclass
class GPTConfig:
    """Configuration for the decoder-only Transformer.

    Attributes:
        vocab_size: Number of tokens in the tokenizer vocabulary.
        embedding_dim: Width of the residual stream.
        num_heads: Number of attention heads in each Transformer block.
        num_layers: Number of stacked Transformer blocks.
        hidden_dim: Intermediate width of each SwiGLU feed-forward network.
        rope_base: Base used to construct RoPE frequencies.
    """

    vocab_size: int
    embedding_dim: int
    num_heads: int
    num_layers: int
    hidden_dim: int
    rope_base: float = 10000.0

In [19]:
config = GPTConfig(
    vocab_size=5000,
    embedding_dim=128,
    num_heads=4,
    num_layers=6,
    hidden_dim=384,
)

print(config)

GPTConfig(vocab_size=5000, embedding_dim=128, num_heads=4, num_layers=6, hidden_dim=384, rope_base=10000.0)


In [20]:
@dataclass
class GPTConfig:
    """Configuration for the decoder-only Transformer.

    Attributes:
        vocab_size: Number of tokens in the tokenizer vocabulary.
        embedding_dim: Width of the residual stream.
        num_heads: Number of attention heads in each Transformer block.
        num_layers: Number of stacked Transformer blocks.
        hidden_dim: Intermediate width of each SwiGLU feed-forward network.
        rope_base: Base used to construct RoPE frequencies.
    """

    vocab_size: int
    embedding_dim: int
    num_heads: int
    num_layers: int
    hidden_dim: int
    rope_base: float = 10000.0

    def __post_init__(self) -> None:
        """Validate relationships between architecture dimensions."""
        if self.embedding_dim % self.num_heads != 0:
            raise ValueError("embedding_dim must be divisible by num_heads.")

        head_dim: int = self.embedding_dim // self.num_heads

        if head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE.")

## 6. Building the Complete GPT Model

The complete decoder-only model contains four major stages:

1. convert token IDs into learned token embeddings,
2. transform the residual stream through a stack of Transformer blocks,
3. normalize the final hidden representations,
4. project every token representation into vocabulary logits.

The shape flow is:

$$
(B,T)
\rightarrow
(B,T,C)
\rightarrow
(B,T,C)
\rightarrow
(B,T,V).
$$

The Transformer stack changes the representation but preserves the
residual-stream shape.

In [22]:
class GPT(nn.Module):
    """A small decoder-only Transformer language model.

    Args:
        config: Architecture configuration for the model.
    """

    def __init__(self, config: GPTConfig) -> None:
        super().__init__()

        self.config = config

        self.token_embedding = nn.Embedding(
            config.vocab_size, config.embedding_dim
        )

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    embedding_dim=config.embedding_dim,
                    attention=CausalSelfAttention(
                        embedding_dim=config.embedding_dim,
                        num_heads=config.num_heads,
                        rope_base=config.rope_base,
                    ),
                    feed_forward=SwiGLU(
                        embedding_dim=config.embedding_dim,
                        hidden_dim=config.hidden_dim,
                    ),
                )
                for _ in range(config.num_layers)
            ]
        )

        self.final_norm = nn.RMSNorm(config.embedding_dim)

        self.lm_head = nn.Linear(
            config.embedding_dim, config.vocab_size, bias=False
        )

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Convert token IDs into next-token logits.

        Args:
            token_ids: Integer token IDs with shape `(B, T)`.

        Returns:
            Vocabulary logits with shape `(B, T, V)`.
        """
        # Convert token IDs to emb.
        x: torch.Tensor = self.token_embedding(token_ids)

        # Each block updates the same `(B, T, C)` residual stream.
        for block in self.blocks:
            x = block(x)

        # Normalize the final residual stream representation.
        x = self.final_norm(x)

        # to vocab id
        logits: torch.Tensor = self.lm_head(x)

        return logits

In [23]:
# Shape test

config = GPTConfig(
    vocab_size=100,
    embedding_dim=32,
    num_heads=4,
    num_layers=3,
    hidden_dim=96,
)

model = GPT(config)


token_ids: torch.Tensor = torch.randint(
    low=0,
    high=config.vocab_size,
    size=(2, 10),
)

print(token_ids.shape)

logits: torch.Tensor = model(token_ids)

print("token IDs:", token_ids.shape)
print("logits:", logits.shape)

torch.Size([2, 10])
token IDs: torch.Size([2, 10])
logits: torch.Size([2, 10, 100])


## 7. Parameter Counting

Before training the model, it is useful to understand where its
parameters are stored.

For the current architecture, parameters mainly come from:

- token embeddings,
- attention projections,
- SwiGLU projections,
- normalization scales,
- and the language-model head.

Counting parameters helps connect the architecture to its memory and
computational cost.

In [24]:
print(
    "total parameters:",
    count_parameters(model),
)

total parameters: 46560


In [25]:
def print_parameter_breakdown(
    model: nn.Module,
) -> None:
    """Print trainable parameter counts for top-level model components.

    Args:
        model: PyTorch module whose registered children should be
            inspected.
    """
    total: int = count_parameters(model)

    for name, module in model.named_children():
        parameter_count: int = count_parameters(module)

        percentage: float = 100.0 * parameter_count / total

        print(f"{name:20s}{parameter_count:8d} ({percentage:5.1f}%)")

    print("-" * 40)
    print(f"{'total':20s}{total:8d}")


print_parameter_breakdown(model)

token_embedding         3200 (  6.9%)
blocks                 40128 ( 86.2%)
final_norm                32 (  0.1%)
lm_head                 3200 (  6.9%)
----------------------------------------
total                  46560


## 8. Whole-Model Causality Verification

Causality must hold across the entire Transformer stack.

Changing a future input token must not change logits produced at earlier
positions.

For a decoder-only model, the representation at position $t$ may depend
only on positions

$$
0,\ldots,t.
$$

This property must remain true after multiple attention layers,
feed-forward networks, residual connections, normalization layers, and
the final language-model head.

In [27]:
model.eval()

token_ids_original: torch.Tensor = torch.tensor(
    [
        [
            5,
            8,
            2,
            9,
            4,
            7,
            3,
            6,
        ]
    ]
)

token_ids_modified: torch.Tensor = token_ids_original.clone()

cutoff: int = 5

token_ids_modified[:, cutoff:] = torch.tensor(
    [
        [
            20,
            21,
            22,
        ]
    ]
)

cutoff: int = 5

token_ids_modified[:, cutoff:] = torch.tensor(
    [
        [
            20,
            21,
            22,
        ]
    ]
)

with torch.no_grad():
    logits_original: torch.Tensor = model(token_ids_original)

    logits_modified: torch.Tensor = model(token_ids_modified)

past_unchanged: bool = torch.allclose(
    logits_original[:, :cutoff, :],
    logits_modified[:, :cutoff, :],
    atol=1e-5,
)

print(
    "past logits unchanged:",
    past_unchanged,
)

future_changed: bool = not torch.allclose(
    logits_original[:, cutoff:, :],
    logits_modified[:, cutoff:, :],
    atol=1e-5,
)

print(
    "future logits changed:",
    future_changed,
)

past logits unchanged: True
future logits changed: True


### Causality Is Preserved Through Depth

Feed-forward networks and normalization operate independently at each
token position.

The only component that mixes information across token positions is
self-attention.

Because every attention layer is causally masked, future information
cannot enter an earlier residual-stream position at any layer.

Therefore, stacking causal Transformer blocks preserves the
autoregressive property of the complete model.

## 9. One End-to-End Training Step

The model architecture is now complete, but a language model is useful
only if gradients can propagate through the entire system.

A single training step follows the pipeline

$$
\text{token IDs}
\rightarrow
\text{GPT}
\rightarrow
\text{logits}
\rightarrow
\text{cross-entropy loss}
\rightarrow
\text{backpropagation}
\rightarrow
\text{parameter update}.
$$

The purpose of this section is not to train a useful model yet.

Instead, we verify that:

- the model produces logits with the expected shape,
- the language-modeling loss can be computed,
- gradients reach the model parameters,
- and an optimizer step actually changes those parameters.

In [28]:
batch_size: int = 4
sequence_length: int = 12

# We need T + 1 tokens to create T input/target pairs.
sequences: torch.Tensor = torch.randint(
    low=0,
    high=config.vocab_size,
    size=(
        batch_size,
        sequence_length + 1,
    ),
)

input_ids: torch.Tensor = sequences[:, :-1]
target_ids: torch.Tensor = sequences[:, 1:]

print("sequences:", sequences.shape)
print("inputs:", input_ids.shape)
print("targets:", target_ids.shape)

sequences: torch.Size([4, 13])
inputs: torch.Size([4, 12])
targets: torch.Size([4, 12])


In [29]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
)

In [ ]:
# `detach()` with a real cope.
parameter_before: torch.Tensor = model.token_embedding.weight.detach().clone()

In [31]:
model.train()

logits: torch.Tensor = model(input_ids)

print("input IDs:", input_ids.shape)
print("logits:", logits.shape)

input IDs: torch.Size([4, 12])
logits: torch.Size([4, 12, 100])


In [32]:
loss: torch.Tensor = F.cross_entropy(
    logits.reshape(
        -1,
        config.vocab_size,
    ),
    target_ids.reshape(-1),
)

print("loss:", loss.item())

loss: 4.758986949920654


In [33]:
random_baseline: float = math.log(config.vocab_size)

print(
    "loss:",
    loss.item(),
)

print(
    "uniform baseline:",
    random_baseline,
)

loss: 4.758986949920654
uniform baseline: 4.605170185988092


In [34]:
optimizer.zero_grad()
loss.backward()

In [35]:
embedding_gradient: torch.Tensor | None = model.token_embedding.weight.grad

print(
    "embedding gradient exists:",
    embedding_gradient is not None,
)

embedding gradient exists: True


In [36]:
if embedding_gradient is not None:
    print(
        "embedding gradient norm:",
        embedding_gradient.norm().item(),
    )

embedding gradient norm: 0.09694142639636993


In [37]:
query_gradient: torch.Tensor | None = model.blocks[
    0
].attention.query_projection.weight.grad

print(
    "query gradient exists:",
    query_gradient is not None,
)

if query_gradient is not None:
    print(
        "query gradient norm:",
        query_gradient.norm().item(),
    )

query gradient exists: True
query gradient norm: 0.07930190116167068


In [39]:
mlp_gradient: torch.Tensor | None = model.blocks[
    0
].feed_forward.up_projection.weight.grad

print(
    "MLP gradient exists:",
    mlp_gradient is not None,
)

if mlp_gradient is not None:
    print(
        "MLP gradient norm:",
        mlp_gradient.norm().item(),
    )

MLP gradient exists: True
MLP gradient norm: 0.2740033268928528


In [40]:
optimizer.step()

In [41]:
parameter_after: torch.Tensor = model.token_embedding.weight.detach().clone()

parameters_changed: bool = not torch.allclose(
    parameter_before,
    parameter_after,
)

print(
    "parameters changed:",
    parameters_changed,
)

parameters changed: True


In [42]:
parameter_change: torch.Tensor = parameter_after - parameter_before

print(
    "maximum absolute change:",
    parameter_change.abs().max().item(),
)

print(
    "parameter-change norm:",
    parameter_change.norm().item(),
)

maximum absolute change: 0.0010294914245605469
parameter-change norm: 0.03392411768436432


In [43]:
model.train()

optimizer.zero_grad()

# Predict the next token at every sequence position.
logits: torch.Tensor = model(input_ids)

# Flatten batch and time because cross-entropy treats each token
# position as one classification example.
loss: torch.Tensor = F.cross_entropy(
    logits.reshape(
        -1,
        config.vocab_size,
    ),
    target_ids.reshape(-1),
)

# Compute gradients through the complete Transformer.
loss.backward()

# Update all trainable model parameters.
optimizer.step()

print(f"training loss: {loss.item():.4f}")

training loss: 4.5300


### End-to-End Training Verification

The complete model now supports the full training path:

$$
(B,T)
\rightarrow
\text{GPT}
\rightarrow
(B,T,V)
\rightarrow
\mathcal{L}
\rightarrow
\nabla_\theta \mathcal{L}
\rightarrow
\theta'.
$$

The important integration checks are:

- the loss is finite,
- gradients propagate into embeddings, attention, and feed-forward
  parameters,
- gradients remain finite,
- and an optimizer step changes the model parameters.

The optimization loop itself is almost unchanged from the earlier
bigram model.

The major increase in complexity lies inside the model's forward pass,
not in the basic gradient-descent procedure.

## Takeaways

This lesson assembled the individual mechanisms from previous chapters
into a complete decoder-only Transformer.

The most important idea is that a Transformer maintains a persistent
**residual stream** with shape

$$
(B,T,C).
$$

Almost everything inside the model can be understood as a learned update
to this stream.

A pre-norm Transformer block performs two such updates:

$$
x
\leftarrow
x
+
\operatorname{Attention}
\left(
\operatorname{RMSNorm}(x)
\right),
$$

followed by

$$
x
\leftarrow
x
+
\operatorname{SwiGLU}
\left(
\operatorname{RMSNorm}(x)
\right).
$$

Although attention and the feed-forward network use different internal
representations, both eventually return to the model dimension $C$ so
that their outputs can be added back to the residual stream.

### Attention changes how tokens communicate

Multi-head attention temporarily reorganizes

$$
(B,T,C)
$$

into

$$
(B,H,T,D),
$$

where

$$
C = HD.
$$

Each head sees the full sequence but operates in its own learned feature
subspace.

RoPE is applied after splitting into heads and before computing
query-key similarities. It rotates queries and keys without changing
their shape:

$$
(B,H,T,D)
\rightarrow
(B,H,T,D).
$$

This allows positional relationships to influence the attention score
directly.

Once scaled dot-product attention had been implemented and understood
explicitly, this lesson could safely use

```python
F.scaled_dot_product_attention(...)
```

as an optimized primitive. This is an important pattern for the rest of
the project:

implement a mechanism explicitly when it is new, then reuse a mature
implementation once its behavior is understood.

The MLP performs a different kind of computation
Attention mixes information between token positions.
SwiGLU instead transforms features within each token independently:
$$(B,\,T,\,C) \rightarrow (B,\,T,\,M) \rightarrow (B,\,T,\,C).$$
The two sublayers therefore play complementary roles:
<pre>
attention  $\rightarrow$ token mixing
SwiGLU     $\rightarrow$ feature transformation
</pre>
Depth changes representations, not their interface
Transformer blocks are stacked sequentially:
$$x_{0} \rightarrow x_{1} \rightarrow \cdots \rightarrow x_{N}.$$
Every layer has its own parameters, but every layer uses the same
external representation shape:
$$(B,\,T,\,C).$$
nn.ModuleList allows these independently parameterized blocks to be
registered and iterated as part of one model.
The complete language model is structurally simple
After the Transformer stack, the model applies a final normalization and
a language-model head:
$$(B,\,T,\,C) \rightarrow (B,\,T,\,V).$$
The complete forward path is therefore
$$(B,\,T) \rightarrow (B,\,T,\,C) \rightarrow (B,\,T,\,C) \rightarrow (B,\,T,\,V).$$
or conceptually:
<pre>
token IDs
(B, T)
    ↓
Token Embedding
    ↓
(B, T, C)
    ↓
Transformer Block × N
    ↓
(B, T, C)
    ↓
Final RMSNorm
    ↓
LM Head
    ↓
logits
(B, T, V)
</pre>
The final logits provide one vocabulary distribution for every sequence
position.
A larger model does not require a different training algorithm
The complete Transformer can be trained with exactly the same
next-token objective studied earlier:
$$\text{token IDs} \rightarrow \text{logits} \rightarrow \text{cross entropy} \rightarrow \text{backpropagation} \rightarrow \text{optimizer step}.$$
The optimization loop is almost unchanged from the earlier bigram
language model.
What became more sophisticated was the function producing the logits.
This is the main architectural lesson of the chapter:

A decoder-only Transformer is a stack of shape-preserving residual
updates that gradually transform token representations until they are
useful for next-token prediction.